# s29 — Redis-backed Mailbox

**What this teaches:** swap the in-memory mailbox from [s26_mailbox](../s26_mailbox/s26_mailbox.ipynb) for a **distributed** one backed by Redis. The agent code is identical except for one environment variable — `MAILBOX_BACKEND=redis`. That is the point: the FSM and tool surface from [s27_fsm](../s27_fsm/s27_fsm.ipynb) are reused unchanged across backends.

**Concept anchor:** distribution matters as soon as agents need to outlive a single process. A Redis-backed mailbox lets a Yoke agent in one container talk to another agent (or a long-running worker, or a human operator's CLI) in a different container or even a different host. The contract — `tell`, `ask`, `check` — does not change.

## Prerequisites

- A working [GoNB](https://github.com/janpfeifer/gonb) kernel.
- An LLM provider configured via env vars before launching Jupyter, e.g.:
  ```
  export YOKE_PROVIDER=openai_compat
  export OPENAI_BASE_URL=http://localhost:11434/v1
  export YOKE_MODEL=qwen2.5-coder
  ```
  Defaults to a local Ollama. Swap in `anthropic` / `openai` / `gemini` for hosted providers — see [docs/providers.md](../../docs/providers.md).
- **A running Redis server**, reachable via `REDIS_URL`, e.g.:
  ```
  export REDIS_URL=redis://localhost:6379/0
  docker run -p 6379:6379 redis:7
  ```
  If `REDIS_URL` is empty this notebook will exit early rather than fall back to the in-memory backend.

## 1. Imports

Identical to [s26_mailbox](../s26_mailbox/s26_mailbox.ipynb).

In [ ]:
import (
    "context"
    "fmt"
    "os"

    "github.com/blouargant/yoke/core/agentkit"
    "github.com/blouargant/yoke/core/stream"
    "github.com/blouargant/yoke/internal/teammates"
)

## 2. Helper

Notebook-safe `must` — panics instead of `os.Exit`.

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Select the Redis backend

`teammates.ChooseBackend()` inspects `MAILBOX_BACKEND` and `REDIS_URL`. Setting both at runtime is enough to flip the implementation under the same interface. We bail early if `REDIS_URL` is unset so the rest of the notebook doesn't silently use the in-memory backend.

In [ ]:
if os.Getenv("REDIS_URL") == "" {
    panic("REDIS_URL not set; this notebook requires a running Redis. See the Prerequisites cell.")
}
os.Setenv("MAILBOX_BACKEND", "redis")
ctx := context.Background()
llm, err := agentkit.NewModel(ctx)
must(err)
be, err := teammates.ChooseBackend()
must(err)
defer be.Close()
me := teammates.NewAgent("lead", be)
fmt.Println("redis-backed mailbox ready for:", "lead")

## 4. Wire and run the agent

Everything below mirrors [s26_mailbox](../s26_mailbox/s26_mailbox.ipynb): same tools, same loop. Only the backend address changed. If you run this notebook twice the mailbox survives between runs — that durability is why distribution matters.

In [ ]:
a, err := agentkit.New(agentkit.AgentConfig{
    Name:  "s29_redis",
    Model: llm,
    Tools: me.Tools(),
})
must(err)
r, err := agentkit.Runner("s29", a)
must(err)
prompt := "Tell teammate 'reviewer' the message 'ping over redis' then check your own mailbox."
must(stream.Print(os.Stdout, agentkit.RunOnce(ctx, r, prompt)))

## What to look for

- The diff against [s26_mailbox](../s26_mailbox/s26_mailbox.ipynb) is *just three lines* (the `REDIS_URL` check and `MAILBOX_BACKEND=redis` set). That is the value of the backend abstraction.
- Open `redis-cli` and run `KEYS yoke:teammates:*` while the notebook is alive — you can literally watch the queues being created.
- The FSM behaviour from [s27_fsm](../s27_fsm/s27_fsm.ipynb) is identical here: ask/reply ordering, single outstanding question, all enforced client-side.

## Try it yourself

1. Start a second process (or another notebook) that runs `teammates.NewAgent("reviewer", be).Check(ctx, 30*time.Second)` against the same Redis. Re-run this notebook and the reviewer instance will receive the message.
2. Kill Redis mid-run and observe the error path — `tell` and `check` surface backend errors directly so the agent (and you) know the connection died.